In [1]:
import gzip, re, random
import pandas as pd

# 1. HMM headers: accession, length, number of seed seqs
rows, cur = [], {}
with gzip.open("Pfam-A.hmm.gz", "rt") as f:
    for line in f:
        if line.startswith("NAME"): cur = {"name": line.split()[1]}
        elif line.startswith("ACC"): cur["acc"] = line.split()[1].split(".")[0]
        elif line.startswith("LENG"): cur["leng"] = int(line.split()[1])
        elif line.startswith("NSEQ"): cur["nseq"] = int(line.split()[1])
        elif line.startswith("//"): rows.append(cur)
hmm = pd.DataFrame(rows)

# 2. Entry type from hmm.dat
types, acc = {}, None
with gzip.open("Pfam-A.hmm.dat.gz", "rt") as f:
    for line in f:
        if line.startswith("#=GF AC"): acc = line.split()[2].split(".")[0]
        elif line.startswith("#=GF TP"): types[acc] = line.split()[2]
hmm["type"] = hmm["acc"].map(types)

# 3. Clans
clans = pd.read_csv("Pfam-A.clans.tsv.gz", sep="\t", header=None,
                    names=["acc","clan","clan_name","fam_name","desc"])
hmm = hmm.merge(clans[["acc","clan"]], on="acc", how="left")

# 4. Apply criteria
sel = hmm[(hmm.nseq >= 110) & (hmm.type == "Family")
          & hmm.leng.between(20, 40)]

# 5. One family per clan (decide how to treat families with no clan)
in_clan = sel[sel.clan.notna()].drop_duplicates("clan")
no_clan = sel[sel.clan.isna()]
print(len(in_clan), len(no_clan))

6 20


In [2]:
FINAL_SET = pd.concat([in_clan, no_clan])
print(len(FINAL_SET))
FINAL_SET.to_csv('protein_families.csv', index=False)

26


In [3]:
"""
Download 100 full-length proteins for each Pfam family in protein_families.csv.
Just edit the settings below and press Run. No command line needed.
"""
import gzip, random, re, time
from pathlib import Path
import pandas as pd
import requests

# ---------------- SETTINGS (edit these) ----------------
FAMILIES_CSV = "protein_families.csv"   # needs columns: acc, name
N_SEQS = 100                            # sequences per family
RNG_SEED = 42                           # keeps the random pick reproducible
OUTPUT_DIR = "sequences"                # one FASTA per family: sequences/<accession>.fasta
# -------------------------------------------------------

INTERPRO = "https://www.ebi.ac.uk/interpro/api/entry/pfam/{acc}/?annotation=alignment:seed"
UNIPROT = "https://rest.uniprot.org/uniprotkb/search"


def get_with_retry(url, params=None):
    for attempt in range(3):
        try:
            r = requests.get(url, params=params, timeout=90)
            if r.status_code == 200:
                return r
            print("   HTTP", r.status_code)
        except requests.RequestException as e:
            print("   error:", e)
        time.sleep(2 * (attempt + 1))
    return None


def get_seed_members(acc):
    """Return a list of (uniprot_id, start, end, domain_sequence) from the seed alignment."""
    r = get_with_retry(INTERPRO.format(acc=acc))
    if r is None:
        return []
    data = r.content
    try:
        data = gzip.decompress(data)
    except OSError:
        pass
    aligned = {}
    for line in data.decode("utf-8", errors="replace").splitlines():
        parts = line.split()
        if len(parts) == 2 and not line.startswith("#") and not line.startswith("//"):
            aligned[parts[0]] = aligned.get(parts[0], "") + parts[1]

    members, seen = [], set()
    for name, seq in sorted(aligned.items()):
        m = re.match(r"^(.+?)/(\d+)-(\d+)$", name)
        domain = seq.replace("-", "").replace(".", "").upper()
        if m and domain and domain not in seen:
            seen.add(domain)
            members.append((re.sub(r"\.\d+$", "", m.group(1)), int(m.group(2)), int(m.group(3)), domain))
    return members


def get_fulllength(ids):
    """Return {uniprot_id: full sequence} for a list of accessions / entry names."""
    terms = [("id:" if "_" in i else "accession:") + i for i in ids]
    r = get_with_retry(UNIPROT, {"query": "(" + " OR ".join(terms) + ")", "format": "fasta", "size": 500})
    found, header, seq = {}, None, []
    if r is None:
        return found
    for line in r.text.splitlines() + [">"]:
        if line.startswith(">"):
            if header:
                parts = header.split("|")
                if len(parts) >= 3:
                    found[parts[1]] = "".join(seq)              # accession
                    found[parts[2].split()[0]] = "".join(seq)   # entry name
            header, seq = line[1:], []
        else:
            seq.append(line.strip())
    return found


families = pd.read_csv(FAMILIES_CSV)
rng = random.Random(RNG_SEED)
out_dir = Path(OUTPUT_DIR)
rows = []

for acc, name in zip(families["acc"], families["name"]):
    print(acc, name)
    members = get_seed_members(acc)
    rng.shuffle(members)

    kept, seen_full, i = [], set(), 0
    while len(kept) < N_SEQS and i < len(members):
        batch = members[i:i + 50]
        i += 50
        full_seqs = get_fulllength([m[0] for m in batch])
        for uid, start, end, domain in batch:
            full = full_seqs.get(uid)
            if full and full not in seen_full:
                seen_full.add(full)
                kept.append((uid, start, end, full))
        time.sleep(0.3)

    kept = kept[:N_SEQS]
    print(f"   {len(kept)} sequences saved (seed members: {len(members)})")

    # one FASTA per family, named by accession
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / f"{acc}.fasta", "w") as f:
        for uid, start, end, full in kept:
            f.write(f">{uid}/{start}-{end}\n{full}\n")

    for uid, start, end, full in kept:
        rows.append({"acc": acc, "family": name, "uniprot_id": uid,
                     "domain_start": start, "domain_end": end, "length": len(full)})

# domain coordinates for every saved protein (so you can trim to the domain later)
table = pd.DataFrame(rows)
table.to_csv(out_dir / "domain_coordinates.csv", index=False)

print("\nDone.")
print(table.groupby("family").size().to_string())

PF24444 DUF7563
   100 sequences saved (seed members: 123)
PF09698 GSu_C4xC__C2xCH
   50 sequences saved (seed members: 165)
PF02375 JmjN
   100 sequences saved (seed members: 257)
PF00467 KOW
   100 sequences saved (seed members: 163)
PF00376 MerR
   100 sequences saved (seed members: 279)
PF03489 SapB_2
   100 sequences saved (seed members: 425)
PF16593 Cas9-BH
   93 sequences saved (seed members: 125)
PF10346 Con-6
   73 sequences saved (seed members: 132)
PF27439 DnaJ_C_2
   100 sequences saved (seed members: 268)
PF06803 DUF1232
   100 sequences saved (seed members: 271)
PF07043 DUF1328
   100 sequences saved (seed members: 126)
PF19124 DUF5808
   100 sequences saved (seed members: 257)
PF19950 DUF6412
   76 sequences saved (seed members: 125)
PF02817 E3_binding
   100 sequences saved (seed members: 683)
PF02260 FATC
   100 sequences saved (seed members: 272)
PF07581 Glug
   18 sequences saved (seed members: 110)
PF16211 Histone_H2A_C
   100 sequences saved (seed members: 125)
PF2

In [4]:
import os

per_acc = pd.DataFrame(table.groupby("acc").size()).reset_index()
per_acc.columns = ['family', 'count']

# delete the families with less than 100
for each in list(per_acc[per_acc['count'] < 100]['family']):
    filepath = os.path.join('sequences', f'{each}.fasta')
    if os.path.isfile(filepath):
        os.remove(filepath)
        print(f"Deleted: {each}")
    else:
        print(f"Not found: {each}")

print(f'There are {len(per_acc[per_acc['count'] == 100]['family'])} remaining.')

Deleted: PF07581
Deleted: PF09698
Deleted: PF10346
Deleted: PF16593
Deleted: PF19950
Deleted: PF27436
There are 20 remaining.
